In [1]:
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
hits = pd.read_parquet("../data/large_files/paul_scan/hits_nlp.parquet", engine="pyarrow")

In [ ]:
def explore_nlp_tokens(nlp_tokens):
    try:
        tokens_flat = [token for sent_data in nlp_tokens for token in sent_data]
        target_indices = [n for n, token in enumerate(tokens_flat) if token[-7]]
        kwic_10target10 = tokens_flat[target_indices[0]-10:target_indices[-1]+10]
        def explore_nlp_tokens(nlp_tokens, window=10,
                               target_words=("paul", "paulus", "pauli")):
            """Return the ±window tokens around the target(s) of a concordance hit.

            Accepts either a nested list-of-sentences of spaCy dicts (hits["kwic_nlp"])
            or a flat list of surface strings (hits["kwic_tokens"]).
            """
            if nlp_tokens is None or len(nlp_tokens) == 0:
                return None
            try:
                # Flatten only if nested; otherwise use the list as-is.
                if isinstance(nlp_tokens[0], list):
                    tokens_flat = [tok for sent in nlp_tokens for tok in sent]
                else:
                    tokens_flat = list(nlp_tokens)
                if not tokens_flat:
                    return None
            except Exception:
                def surface(tok):
                    if isinstance(tok, dict):
                        for k in ("text", "token", "lower", "orth"):                            if tok.get(k) not in (None, ""):                                return str(tok[k])                        return ""                    return str(tok)                def is_target(tok):                    if isinstance(tok, dict):                        for k in ("is_target", "target", "is_match", "match"):                            if tok.get(k):                                return True                    cleaned = surface(tok).strip(".,;:!?()[]{}\"'\u201c\u201d\u2018\u2019").lower()                    return cleaned in target_words                target_indices = [n for n, tok in enumerate(tokens_flat) if is_target(tok)]
                if not target_indices:
                    return None


In [9]:
hits["kwic_10target10"] = hits["kwic_tokens"].apply(explore_nlp_tokens)

In [13]:
hits.sample(10)["kwic_10target10"] 

158805    None
108796    None
206626    None
99357     None
278709    None
323119    None
16849     None
300301    None
411491    None
131229    None
Name: kwic_10target10, dtype: object